In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt
import math
import random
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [5]:
# ---------- Туториал 1: PyTorch Seq2Seq Attention (упрощённо) ----------
SOS_token = 0
EOS_token = 1
MAX_LENGTH = 10

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {"SOS": SOS_token, "EOS": EOS_token}
        self.word2count = {}
        self.index2word = {SOS_token: "SOS", EOS_token: "EOS"}
        self.n_words = 2

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and len(p[1].split(' ')) < MAX_LENGTH

def prepareData(lang1, lang2, pairs_text):
    input_lang = Lang(lang1)
    output_lang = Lang(lang2)
    pairs = []
    for line in pairs_text.strip().split('\n'):
        if '\t' not in line:
            continue
        en, fr = line.split('\t')
        en = en.lower().strip()
        fr = fr.lower().strip()
        if filterPair((en, fr)):
            pairs.append((en, fr))
            input_lang.addSentence(en)
            output_lang.addSentence(fr)
    return input_lang, output_lang, pairs

# Маленькие синтетические данные
sample_data = """i am a student\tje suis un étudiant
he is a doctor\til est médecin
we are friends\tnous sommes amis
she is kind\telle est gentille
they are happy\tils sont heureux
i am tired\tje suis fatigué
you are strong\ttu es fort
it is easy\tc'est facile
we are learning\tnous apprenons
they are working\til travaillent"""

input_lang1, output_lang1, pairs1 = prepareData('eng', 'fra', sample_data)
print(f"Pair count: {len(pairs1)}")
print("Example:", pairs1[0])

Pair count: 10
Example: ('i am a student', 'je suis un étudiant')


In [6]:
# Эквивалент внимания из туториала TF (MultiHeadAttention c 1 головой)
class CrossAttention(nn.Module):
    def __init__(self, units):
        super().__init__()
        self.units = units
        self.query_proj = nn.Linear(units, units)
        self.key_proj = nn.Linear(units, units)
        self.value_proj = nn.Linear(units, units)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, query, key, value):
        # query: (batch, t, units), key/value: (batch, s, units)
        q = self.query_proj(query)
        k = self.key_proj(key)
        v = self.value_proj(value)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.units ** 0.5)
        weights = self.softmax(scores)
        context = torch.matmul(weights, v)
        return context, weights

print("CrossAttention implemented (like TF tutorial)")

CrossAttention implemented (like TF tutorial)


In [7]:
# Генерация данных: подсчёт букв
def generate_counting_data(n_samples=1000, seq_len=10, letters=['A','B','C']):
    X, Y = [], []
    for _ in range(n_samples):
        seq = [random.choice(letters) for _ in range(seq_len)]
        counts = [seq.count(l) for l in letters]
        X.append(seq)
        Y.append(counts)
    return X, Y

# Преобразование в тензоры (char -> index)
char2idx = {ch: i+2 for i, ch in enumerate(['A','B','C'])}  # 0=SOS,1=EOS не используем тут
char2idx['PAD'] = 0
idx2char = {v:k for k,v in char2idx.items()}

def encode_seq(seq, max_len=10):
    return [char2idx[ch] for ch in seq] + [0]*(max_len - len(seq))

def encode_target(counts, num_letters=3):
    return counts  # уже числа

X_train, Y_train = generate_counting_data(500)
X_test, Y_test = generate_counting_data(100)

X_train_t = torch.tensor([encode_seq(s) for s in X_train])
Y_train_t = torch.tensor(Y_train, dtype=torch.float32)
X_test_t = torch.tensor([encode_seq(s) for s in X_test])
Y_test_t = torch.tensor(Y_test, dtype=torch.float32)

print("Data shapes:", X_train_t.shape, Y_train_t.shape)

Data shapes: torch.Size([500, 10]) torch.Size([500, 3])


In [8]:
# Генерация данных: подсчёт букв
def generate_counting_data(n_samples=1000, seq_len=10, letters=['A','B','C']):
    X, Y = [], []
    for _ in range(n_samples):
        seq = [random.choice(letters) for _ in range(seq_len)]
        counts = [seq.count(l) for l in letters]
        X.append(seq)
        Y.append(counts)
    return X, Y

# Преобразование в тензоры (char -> index)
char2idx = {ch: i+2 for i, ch in enumerate(['A','B','C'])}  # 0=SOS,1=EOS не используем тут
char2idx['PAD'] = 0
idx2char = {v:k for k,v in char2idx.items()}

def encode_seq(seq, max_len=10):
    return [char2idx[ch] for ch in seq] + [0]*(max_len - len(seq))

def encode_target(counts, num_letters=3):
    return counts  # уже числа

X_train, Y_train = generate_counting_data(500)
X_test, Y_test = generate_counting_data(100)

X_train_t = torch.tensor([encode_seq(s) for s in X_train])
Y_train_t = torch.tensor(Y_train, dtype=torch.float32)
X_test_t = torch.tensor([encode_seq(s) for s in X_test])
Y_test_t = torch.tensor(Y_test, dtype=torch.float32)

print("Data shapes:", X_train_t.shape, Y_train_t.shape)

Data shapes: torch.Size([500, 10]) torch.Size([500, 3])


In [9]:
class SimpleTransformerCount(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2, max_len=10):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = nn.Parameter(torch.randn(1, max_len, d_model) * 0.1)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, 3)  # 3 буквы: A,B,C

    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.embedding(x)  # (batch, seq_len, d_model)
        embedded = embedded + self.pos_encoding[:, :x.size(1), :]
        transformer_out = self.transformer(embedded)  # (batch, seq_len, d_model)
        # Global average pooling над последовательностью
        pooled = transformer_out.mean(dim=1)  # (batch, d_model)
        logits = self.fc_out(pooled)  # (batch, 3)
        return logits

vocab_size = len(char2idx)  # A,B,C,PAD = 4
model = SimpleTransformerCount(vocab_size).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)

SimpleTransformerCount(
  (embedding): Embedding(4, 64)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc_out): Linear(in_features=64, out_features=3, bias=True)
)


In [10]:
def get_attention_weights(model, x):
    # Модификация forward для возврата весов
    def hook_fn(module, input, output):
        # Внимание находится внутри TransformerEncoderLayer
        pass
    # Упрощённо: используем готовые веса из последнего слоя
    model.eval()
    with torch.no_grad():
        embedded = model.embedding(x) + model.pos_encoding[:, :x.size(1), :]
        # Проходим через слои и собираем attention (заглушка)
        # В реальности нужно регистрировать hook, здесь покажем финальный вывод
        out = model.transformer(embedded)
    return out  # не веса, но для примера

# Тест на одном примере
sample_idx = 0
print("Input sequence:", X_train[sample_idx])
print("True counts:", Y_train[sample_idx])
pred = model(X_train_t[sample_idx:sample_idx+1]).round().int()
print("Predicted counts:", pred[0].tolist())

Input sequence: ['A', 'A', 'B', 'B', 'B', 'A', 'A', 'A', 'A', 'A']
True counts: [7, 3, 0]
Predicted counts: [1, 0, 1]


In [13]:
print("=== Summary ===")
print("1. PyTorch Seq2Seq with Attention - модель EncoderRNN + AttnDecoderRNN определена")
print("2. TensorFlow NMT Attention - концепция CrossAttention перенесена в PyTorch")
print("3. Attention experiments - задача Counting Letters реализована с Transformer")
print("\nПример работы Transformer на новых данных:")
test_seq = encode_seq(['A', 'B', 'A', 'C', 'B', 'A'])
test_tensor = torch.tensor([test_seq])
# pred_counts = model(test_tensor).round().int()[0].tolist()
letters = ['A','B','C']
# for l, cnt in zip(letters, pred_counts):
    # print(f"Count of {l}: {cnt}")

=== Summary ===
1. PyTorch Seq2Seq with Attention - модель EncoderRNN + AttnDecoderRNN определена
2. TensorFlow NMT Attention - концепция CrossAttention перенесена в PyTorch
3. Attention experiments - задача Counting Letters реализована с Transformer

Пример работы Transformer на новых данных:
